# 02 Preprocessing and PCA

This notebook prepares the cleaned GSE40732 expression matrix for exploratory dimensionality reduction. It removes near-zero variance features, standardizes expression values, and visualizes samples with PCA. No classification models are trained here.

## 1. Setup

Import Python packages and define project paths.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "figures"
results_dir = project_root / "results"

expression_path = processed_dir / "gse40732_expression.csv"
labels_path = processed_dir / "gse40732_labels.csv"
pca_coordinates_path = results_dir / "gse40732_pca_coordinates.csv"
pca_plot_path = figures_dir / "gse40732_pca_plot.png"

figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Expression Data and Labels

Load the cleaned expression matrix and sample labels produced by the previous workflow steps.

In [ ]:
for path in [expression_path, labels_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

expression = pd.read_csv(expression_path, index_col=0)
labels = pd.read_csv(labels_path, index_col=0)

expression = expression.apply(pd.to_numeric, errors="coerce")

print(f"Raw expression shape: {expression.shape}")
print(f"Labels shape: {labels.shape}")
display(labels.head())

## 3. Orient Matrix and Align Samples

Machine learning workflows usually expect samples as rows and features as columns. If the expression matrix is stored as genes/probes by samples, transpose it and then align it to the label table by sample ID.

In [ ]:
label_sample_ids = labels.index.astype(str)
row_matches = expression.index.astype(str).isin(label_sample_ids).sum()
column_matches = expression.columns.astype(str).isin(label_sample_ids).sum()

if column_matches > row_matches:
    X = expression.T.copy()
    print("Expression matrix appeared to be genes/probes x samples; transposed to samples x features.")
else:
    X = expression.copy()
    print("Expression matrix appeared to already be samples x features.")

X.index = X.index.astype(str)
labels.index = labels.index.astype(str)

common_samples = X.index.intersection(labels.index)
if common_samples.empty:
    raise ValueError("No overlapping sample IDs found between expression matrix and labels.")

X = X.loc[common_samples].sort_index()
y = labels.loc[common_samples, "label"].sort_index()

print(f"Aligned samples: {len(common_samples)}")
print(f"Final feature matrix shape before filtering: {X.shape}")
print("Class counts:")
print(y.value_counts())

## 4. Remove Near-Zero Variance Features

Features with no variance do not help PCA and can interfere with scaling. `VarianceThreshold` removes constant or near-constant features using a simple variance cutoff.

In [ ]:
variance_threshold = 1e-8
selector = VarianceThreshold(threshold=variance_threshold)
X_filtered_array = selector.fit_transform(X)

kept_features = X.columns[selector.get_support()]
removed_features = X.shape[1] - len(kept_features)
X_filtered = pd.DataFrame(X_filtered_array, index=X.index, columns=kept_features)

print(f"Variance threshold: {variance_threshold}")
print(f"Features before filtering: {X.shape[1]}")
print(f"Features after filtering: {X_filtered.shape[1]}")
print(f"Features removed by variance filtering: {removed_features}")

## 5. Standardize Features

Standardize each feature to mean zero and unit variance before PCA so high-scale expression features do not dominate the components.

In [ ]:
if X_filtered.isna().any().any():
    missing_total = int(X_filtered.isna().sum().sum())
    raise ValueError(
        f"Expression matrix contains {missing_total} missing values after filtering. "
        "Handle missing values before PCA."
    )

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

print(f"Scaled matrix shape: {X_scaled.shape}")

## 6. Run PCA

Fit a two-component PCA model and save the sample coordinates for downstream reporting.

In [ ]:
pca = PCA(n_components=2, random_state=3888)
pca_coords = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    pca_coords,
    index=X_filtered.index,
    columns=["PC1", "PC2"],
)
pca_df.index.name = "sample_id"
pca_df["label"] = y.loc[pca_df.index].values

pc1_var, pc2_var = pca.explained_variance_ratio_
print(f"PC1 explained variance ratio: {pc1_var:.4f}")
print(f"PC2 explained variance ratio: {pc2_var:.4f}")

pca_df.to_csv(pca_coordinates_path)
print(f"Saved PCA coordinates to {pca_coordinates_path}")
display(pca_df.head())

## 7. Plot PCA Scatter

Plot the first two principal components colored by asthma/control label.

In [ ]:
colors = {"Control": "#4C78A8", "Asthma": "#F58518"}

fig, ax = plt.subplots(figsize=(7, 5))
for label, group in pca_df.groupby("label"):
    ax.scatter(
        group["PC1"],
        group["PC2"],
        label=label,
        color=colors.get(label, "#999999"),
        alpha=0.8,
        edgecolor="white",
        linewidth=0.4,
    )

ax.set_title("GSE40732 PCA")
ax.set_xlabel(f"PC1 ({pc1_var * 100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pc2_var * 100:.1f}% variance)")
ax.legend(title="Class")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()
fig.savefig(pca_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved PCA plot to {pca_plot_path}")